# LocalSparse · Milestone 1 (Colab A100)

Goal: take a pretrained 1B base (MiniCPM5-1B), surgically replace its dense attention with our 3-branch sparse attention, run 200 'healing' steps on FineWeb, and confirm PPL stays within 1.3× the base model.

Expected wall-clock on A100: ~30–60 min. Budget cap from spec §2.9: **$5**.

If this run satisfies the PPL gate, proceed to M2 (indexer + selected branch alive).

## 0. Environment

In [ ]:
!nvidia-smi || true
!pip -q install -U pip
!git clone https://github.com/kaaninel/localsparse.git || (cd localsparse && git pull)
%cd localsparse
!pip -q install -e '.[training,unsloth]'

## 1. Sanity — run the test suite

In [ ]:
!pytest tests/ -x -q

## 2. Download MiniCPM5-1B (cached)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
BASE = 'openbmb/MiniCPM5-1B'
tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16, trust_remote_code=True)
print(base.config)

## 3. Establish baseline PPL

In [ ]:
import torch, math
from datasets import load_dataset
ds = load_dataset('HuggingFaceFW/fineweb', split='train', streaming=True)
EVAL_TOKENS = 50_000
def ppl(model):
    model.eval(); model = model.cuda()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for ex in ds:
            ids = tok(ex['text'], return_tensors='pt', truncation=True, max_length=1024).input_ids.cuda()
            if ids.shape[1] < 8: continue
            out = model(ids, labels=ids)
            total_loss += float(out.loss) * ids.shape[1]
            total_tokens += ids.shape[1]
            if total_tokens > EVAL_TOKENS: break
    return math.exp(total_loss / total_tokens)
base_ppl = ppl(base)
print(f'BASE PPL = {base_ppl:.4f}')

## 4. Surgery

In [ ]:
from localsparse.config import default_config
from localsparse.model.surgery import perform_surgery, detect_model_dims
cfg = default_config()
cfg.model = detect_model_dims(base.config)
report = perform_surgery(base, cfg)
print(report)
base.cuda()

## 5. M1 healing run

In [ ]:
from localsparse.training import M1Config, run_m1
from localsparse.training.data import FineWebStream
m1_cfg = M1Config(steps=200, batch_size=1, seq_len=1024, lr=5e-5,
                  branch_balance_weight=0.01, surgery_kl_weight=0.0, log_every=10)
stream = FineWebStream(tok, seq_len=1024, batch_size=1)
stats = run_m1(base, teacher=None, batch_iter=stream, cfg=m1_cfg)

## 6. Post-surgery PPL

In [ ]:
post_ppl = ppl(base)
ratio = post_ppl / base_ppl
print(f'POST PPL = {post_ppl:.4f}  (ratio={ratio:.3f}x)')
assert ratio <= 1.3, f'M1 FAILED: PPL ratio {ratio:.3f} > 1.3'
print('M1 PASSED ✅')

## 7. Save

In [ ]:
import json, pathlib
out = pathlib.Path('/content/drive/MyDrive/localsparse_m1')
out.mkdir(parents=True, exist_ok=True)
base.save_pretrained(out)
tok.save_pretrained(out)
(out / 'm1_stats.json').write_text(json.dumps({
    'base_ppl': base_ppl, 'post_ppl': post_ppl, 'ratio': ratio,
    'final_branch_mass': stats.branch_mass_history[-1],
    'layers_replaced': report.layers_replaced,
}, indent=2))
print('saved to', out)